<center>
  <h1><b>SeedUp - Smart Torrent Management V1</b></h1>

---


### **Important Notes:**

1. **Free Tier Limitations:** Google Colab free tier has:
   - Limited runtime
   - Limited disk space (~100GB)
   - Session may disconnect if idle

2. **Legal Usage Only:** Only download content you have the right to download.

3. **Google Drive Space:** Ensure you have enough space in your Google Drive.

---
# **📥 Step 1: Clone Project Files**

Download the torrent downloader scripts from GitHub.

In [ ]:
#@title **Clone Your Modified Repository** { display-mode: "form" }
#@markdown Fork SeedUp, apply the selective-download changes, then paste your fork URL here.
REPOSITORY_URL = "https://github.com/YOUR_USERNAME/SeedUp.git" #@param {type:"string"}

import os
import shutil
import subprocess

PROJECT_DIR = "SeedUp"

if "YOUR_USERNAME" in REPOSITORY_URL or not REPOSITORY_URL.strip():
    raise ValueError("Replace REPOSITORY_URL with the URL of your modified GitHub fork.")

if os.path.exists(PROJECT_DIR):
    shutil.rmtree(PROJECT_DIR)

subprocess.run(["git", "clone", REPOSITORY_URL.strip(), PROJECT_DIR], check=True)
os.chdir(PROJECT_DIR)

print("\n✅ Modified project files downloaded successfully!")


---
# **🔧 Step 2: Install Dependencies**

This will install all required packages including libtorrent and Google Drive API clients.

In [ ]:
#@title **Install Required Packages** { display-mode: "form" }
#@markdown Click the ▶️ button to install dependencies (takes ~2 minutes)

print("📦 Installing dependencies...\n")

# Install libtorrent
!pip install -r requirements.txt > /dev/null 2>&1


print("\n✅ All dependencies installed successfully!")
print("📌 Ready to proceed to the next step.")

In [ ]:
#@title **Verify Installations** { display-mode: "form" }
#@markdown Click the ▶️ button to verify all installations are working properly

import sys
import subprocess

def verify_installation():
    print("🔍 Verifying installations...\n")

    # Check Python version
    py_version = sys.version.split()[0]
    print(f"📌 Python Version: {py_version}")

    # Check libtorrent
    try:
        import libtorrent as lt
        print(f"📌 libtorrent Version: {lt.version}")
    except ImportError:
        print("❌ libtorrent not found!")
        return False

    # Check Google API clients
    try:
        from googleapiclient import discovery
        from google.oauth2.credentials import Credentials
        print("📌 Google API clients: ✓")
    except ImportError:
        print("❌ Google API clients not found!")
        return False

    print("\n✅ All dependencies verified successfully!")
    return True

verify_installation()

# **🔐 Step 3: Authenticate with Google Drive**

This will open a popup to authenticate your Google account and grant access to Google Drive.

In [ ]:
#@title **Authenticate Google Drive** { display-mode: "form" }
#@markdown Click ▶️ and follow the authentication prompts

from google.colab import auth
auth.authenticate_user()

print("✅ Google Drive authentication successful!")
print("📌 You can now upload files to your Google Drive.")

# **🌐 Step 5A: View Files Inside the Torrent**

Run this first. It fetches the torrent metadata and prints every file with a number.

Use those numbers in Step 5B, for example: `1,3,7-10`.


In [ ]:
#@title **List Torrent Files** { display-mode: "form" }

#@markdown Paste a magnet link OR path to a `.torrent` file:
TORRENT_SOURCE = "" #@param {type:"string"}

import subprocess

if not TORRENT_SOURCE.strip():
    print("⚠️ Please enter a magnet link or torrent file path!")
else:
    subprocess.run([
        "python", "main.py", "list-files",
        "-t", TORRENT_SOURCE.strip()
    ], check=False)


# **🌐 Step 5B: Download Selected Files**

Enter the numbers shown in Step 5A.

- Example: `1,3,7-10`
- Leave **Selected Files** blank to download the complete torrent.
- Only the selected files will be retained and uploaded to Google Drive.


In [ ]:
#@title **Download Selected Torrent Files** { display-mode: "form" }

#@markdown ### 🔗 Torrent Information
TORRENT_SOURCE = "" #@param {type:"string"}

#@markdown ### 📄 Selected Files
#@markdown Use file numbers from Step 5A, such as `1,3,7-10`. Blank means all files.
SELECTED_FILES = "" #@param {type:"string"}

#@markdown ### ⚙️ Options
AUTO_UPLOAD = True #@param {type:"boolean"}
SKIP_EXISTING = True #@param {type:"boolean"}

import subprocess

if not TORRENT_SOURCE.strip():
    print("⚠️ Please enter a magnet link or torrent file path!")
else:
    cmd = [
        "python", "main.py", "download",
        "-t", TORRENT_SOURCE.strip()
    ]

    if SELECTED_FILES.strip():
        cmd.extend(["--files", SELECTED_FILES.strip()])

    if AUTO_UPLOAD:
        cmd.append("--upload")
        print("📁 Selected files will be uploaded to 'SeedUp Downloads'.\n")

    if not SKIP_EXISTING:
        cmd.append("--no-skip")

    subprocess.run(cmd, check=False)


# **📤 Step 6: Upload Existing Files**

If you already have downloaded files and want to upload them to Google Drive separately.

**Note:** Files will be uploaded to the 'SeedUp' folder automatically (or your custom folder if specified)

In [ ]:
#@title **Upload Files to Drive** { display-mode: "form" }

#@markdown ### 📁 File/Folder to Upload
LOCAL_PATH = "" #@param {type:"string"}

#@markdown ### ⚙️ Options
SKIP_EXISTING_FILES = True #@param {type:"boolean"}

import os

if not LOCAL_PATH:
    print("⚠️ Please enter a file or folder path to upload!")
elif not os.path.exists(LOCAL_PATH):
    print(f"⚠️ Path does not exist: {LOCAL_PATH}")
else:
    print("📤 Starting upload to Google Drive...\n")
    print("📁 Files will be uploaded to 'SeedUp Downloads' folder in Drive root\n")

    # Build command
    cmd = f"python main.py upload -p \"{LOCAL_PATH}\""

    if not SKIP_EXISTING_FILES:
        cmd += " --no-skip"

    # Execute upload
    !{cmd}

# **🔍 Step 7: Check Download Status**

Check if there's a paused download that can be resumed.

In [ ]:
#@title **Check Status** { display-mode: "form" }

!python main.py status

# **🧹 Step 8: Clear Session (Optional)**

Clear any saved download session if you want to start fresh.

In [ ]:
#@title **Clear Download Session** { display-mode: "form" }

!python main.py clear


---

## 📚 **Additional Information**

### Troubleshooting

**Download is slow:**
- This depends on the number of seeders and your torrent health
- Colab's network speed varies
- Try using different trackers or magnet links

**Upload failing:**
- Check your Google Drive has enough space
- Re-authenticate if needed
- Check if files already exist when using skip option
- The SeedUp folder is created automatically - no manual setup needed

**Session disconnected:**
- Downloads are saved and can be resumed
- Use the status check to verify saved sessions
- Run the download command again with the same torrent
- Clear stuck sessions if needed

### Best Practices

1. **For large downloads:**
   - Monitor the session and be ready to resume if it disconnects
   - Use the keep-alive cell to prevent idle disconnects
   - Split very large downloads into smaller parts

2. **Storage management:**
   - All files are organized in the 'SeedUp Downloads' folder automatically
   - Clear the downloads folder after successful upload
   - Monitor Colab's disk space usage
   - Use `skip-existing` to avoid duplicate uploads

3. **Finding your uploads:**
   - Open Google Drive and look for the "SeedUp Downloads" folder
   - All uploaded files will be there
   - The folder link is displayed after each upload

### Session Management

The project uses a sophisticated session management system that:
- Saves download progress automatically
- Enables resuming from the exact point of interruption
- Maintains tracker and peer information
- Cleans up automatically after successful completion

---
<br>

### 🔗 **Links**

- [GitHub Repository](https://github.com/codercyco/SeedUp)
- [Report Issues](https://github.com/codercyco/SeedUp/issues)

<br>

---

<p><b>Liked it? You can buy me a coffee ☕</b></p>
<a href="https://www.buymeacoffee.com/codercyco" target="_blank">
  <img src="https://cdn.buymeacoffee.com/buttons/v2/default-blue.png" alt="Buy Me A Coffee" width="150">
</a>

<br>

---

<center>
<p>Made with ❤️ for the community</p>
<p>Created by <b>Ishara Deshapriya</b></p>

---